### New Features 



##### Potential new features : 
- () artist popularity score https://groover.co/en/lp/free-tools/spotify-popularity-score/
- (DONE) Listeners or followers : https://kworb.net/spotify/listeners.html )(caveats : the listeners are the current listeners. not the lisenters at the time when the song was on the chart.) 
- (DONE) Collab / Featured artist (how many artists are on the song) 
    -  in the case when there are multiple artists, maybe can consider taking the max ? 

**unsure**
- Label (major or not, could be related to marketing effort)
- Genre finder : https://www.submithub.com/whats-my-genre
- Music video (dummy)
- album sales (songs that are realeased as single will have no data)


In [ ]:
import pandas as pd
import numpy as np
import lxml
import requests

In [2]:
df = pd.read_csv("/Users/z/Downloads/Spotify-Trend-Analyzer/final_songs_with_audio_features.csv")

In [ ]:
def count_artists(artist_names):
    """Count the number of artists in the artist_name column."""
    if pd.isna(artist_names):
        return 0
    
    if "," in artist_names:
        artists = artist_names.strip('"').split(',')
        return len(artists)

    else : 
        return 1 #single artist


df['artist_count'] = df['artist_names'].apply(count_artists)
df['collaboration_type'] = np.where(df['artist_count'] > 1, 1, 0) # 1 is collab, 0 is solo

In [ ]:
def get_kworb_listeners():
    """Scrape the full listeners table from kworb into a dict {artist: listeners}."""
    all_data = {}
    for i in range(1, 6):  
        url = "https://kworb.net/spotify/listeners.html" if i == 1 else f"https://kworb.net/spotify/listeners{i}.html"
        try:
            tables = pd.read_html(url)
            df = tables[0][["Artist", "Listeners"]]
            df["Listeners"] = df["Listeners"].astype(str).str.replace(",", "").astype(int)
            for _, row in df.iterrows():
                all_data[row["Artist"].lower()] = row["Listeners"]
        except Exception:
            break
    return all_data

def lookup_listeners(artist_name, listeners_dict):
    return listeners_dict.get(artist_name.strip('"').lower(), None)

listeners_dict = get_kworb_listeners()

df["monthly_listeners"] = df["artist_names"].apply(
    lambda x: lookup_listeners(x.strip('"').split(",")[0].strip(), listeners_dict) #take the first/ primary artist
)

In [ ]:
# find primary artist's popularity score

popularity_df = pd.read_csv("artist_popularity_data.csv")
def find_primary_artist(artist_names):
    """find primary artist"""
    if "," in artist_names:
        artists = artist_names.strip('"').split(',')
        primary_artist =  artists[0] #primary artist
    else : 
        primary_artist = artist_names
    return primary_artist

df['primary_artist'] = df['artist_names'].apply(find_primary_artist)

merged_df = pd.merge(df, popularity_df, left_on = 'primary_artist', right_on = 'artist_name', how = 'left')
merged_df.drop(columns = ['artist_name', 'primary_artist'], inplace = True)
merged_df.reset_index(drop = True, inplace = True)


In [ ]:
#export 
merged_df.to_csv('new_data.csv', index=False)

In [ ]:
#check
new_df = pd.read_csv('new_data.csv')
new_df.columns

In [ ]:
new_df.isna().sum()